In [22]:
# Import Libraries

import pandas as pd
from sklearn.preprocessing import LabelEncoder


# Load Dataset

df = pd.read_csv("api_errors_logs.csv")


# Data Preprocessing

df = df.drop_duplicates()
df = df.dropna()


# Create Target Variable

failure_codes = [408, 429, 500, 502, 503]

df['failure'] = df['status_code'].apply(
    lambda x: 1 if x in failure_codes else 0
)


# Remove Irrelevant Features

df = df.drop(
    columns=['request_id', 'endpoint'],
    errors='ignore'
)


# Process Timestamp Features

if 'timestamp' in df.columns:

    df['timestamp'] = pd.to_datetime(df['timestamp'])

    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek

    df = df.drop(columns=['timestamp'])


# Encode Categorical Variables

label_encoder = LabelEncoder()

categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    df[col] = label_encoder.fit_transform(df[col].astype(str))


# Save Cleaned Dataset

df.to_csv("api_errors_logs_cleaned.csv", index=False)

print("Cleaned dataset saved successfully")

Cleaned dataset saved successfully


In [19]:
# Xgboost Baseline

# Import Libraries

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

from xgboost import XGBClassifier
from xgboost import plot_importance


# Load Dataset

df = pd.read_csv("api_errors_logs.csv")


# Initial Inspection

print("Dataset Shape:", df.shape)
print(df.head())
print(df.info())


# Data Preprocessing

df = df.drop_duplicates()
df = df.dropna()


# Create Target Variable

failure_codes = [408, 429, 500, 502, 503]

df['failure'] = df['status_code'].apply(
    lambda x: 1 if x in failure_codes else 0
)


# Remove Irrelevant Features

drop_cols = ['request_id', 'endpoint']

df = df.drop(
    columns=[col for col in drop_cols if col in df.columns],
    errors='ignore'
)


# Process Timestamp Features

if 'timestamp' in df.columns:

    df['timestamp'] = pd.to_datetime(df['timestamp'])

    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek

    df = df.drop(columns=['timestamp'])


# Encode Categorical Variables

label_encoder = LabelEncoder()

categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    df[col] = label_encoder.fit_transform(df[col].astype(str))


# Split Features and Target

X = df.drop(columns=['failure', 'status_code'], errors='ignore')
y = df['failure']


# Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    stratify=y,
    random_state=42
)


# Build XGBoost Model

model = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)


# Train Model

model.fit(X_train, y_train)


# Make Predictions

predictions = model.predict(X_test)


# Model Evaluation

accuracy = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print("\nModel Results")
print("Accuracy:", accuracy)
print("F1 Score:", f1)

print("\nClassification Report")
print(classification_report(y_test, predictions))

Dataset Shape: (220000, 22)
             timestamp       api_name service_owner environment http_method  \
0  2024-01-01 00:00:00  inventory-api     team-beta         dev      DELETE   
1  2024-01-01 00:00:05       user-api    team-alpha     staging      DELETE   
2  2024-01-01 00:00:10      order-api    team-alpha        prod      DELETE   
3  2024-01-01 00:00:15       user-api     team-beta        prod         GET   
4  2024-01-01 00:00:20       user-api    team-gamma        prod         GET   

     endpoint  status_code   error_type                   root_cause  \
0  /v1/lljugd          503      Timeout      High latency in network   
1  /v1/mckssy          500  ClientError  Database connection failure   
2  /v1/ejvxxt          401    AuthError      Schema validation error   
3  /v1/noqarz          400    RateLimit  Database connection failure   
4  /v1/rcqwsq          401  ClientError  Missing authorization scope   

   latency_ms  ...  retry_count  is_retry_successful        clie

In [7]:
# Random Forest Baseline
# Import Libraries

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier


# Load Dataset

df = pd.read_csv("api_errors_logs.csv")


# Initial Inspection

print("Dataset Shape:", df.shape)
print(df.head())
print(df.info())


# Data Preprocessing

df = df.drop_duplicates()
df = df.dropna()


# Create Target Variable

failure_codes = [408, 429, 500, 502, 503]

df['failure'] = df['status_code'].apply(
    lambda x: 1 if x in failure_codes else 0
)


# Remove Irrelevant Features

drop_cols = ['request_id', 'endpoint']

df = df.drop(
    columns=[col for col in drop_cols if col in df.columns],
    errors='ignore'
)


# Process Timestamp Features

if 'timestamp' in df.columns:

    df['timestamp'] = pd.to_datetime(df['timestamp'])

    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek

    df = df.drop(columns=['timestamp'])


# Encode Categorical Variables

label_encoder = LabelEncoder()

categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    df[col] = label_encoder.fit_transform(df[col].astype(str))


# Split Features and Target

X = df.drop(columns=['failure', 'status_code'], errors='ignore')
y = df['failure']


# Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    stratify=y,
    random_state=42
)


# Build Random Forest Model

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)


# Train Model

model.fit(X_train, y_train)


# Make Predictions

predictions = model.predict(X_test)


# Model Evaluation

accuracy = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print("\nModel Results")
print("Accuracy:", accuracy)
print("F1 Score:", f1)

print("\nClassification Report")
print(classification_report(y_test, predictions))

Dataset Shape: (220000, 22)
             timestamp       api_name service_owner environment http_method  \
0  2024-01-01 00:00:00  inventory-api     team-beta         dev      DELETE   
1  2024-01-01 00:00:05       user-api    team-alpha     staging      DELETE   
2  2024-01-01 00:00:10      order-api    team-alpha        prod      DELETE   
3  2024-01-01 00:00:15       user-api     team-beta        prod         GET   
4  2024-01-01 00:00:20       user-api    team-gamma        prod         GET   

     endpoint  status_code   error_type                   root_cause  \
0  /v1/lljugd          503      Timeout      High latency in network   
1  /v1/mckssy          500  ClientError  Database connection failure   
2  /v1/ejvxxt          401    AuthError      Schema validation error   
3  /v1/noqarz          400    RateLimit  Database connection failure   
4  /v1/rcqwsq          401  ClientError  Missing authorization scope   

   latency_ms  ...  retry_count  is_retry_successful        clie